In [372]:
import pandas as pd
import re
import quopri

In [373]:
emails_df = pd.read_csv('../data/01_extracted_emails.csv')
print(emails_df.head(5))

                                    Message-ID  \
0  18782981.1075855378110.JavaMail.evans@thyme   
1  15464986.1075855378456.JavaMail.evans@thyme   
2  24216240.1075855687451.JavaMail.evans@thyme   
3  13505866.1075863688222.JavaMail.evans@thyme   
4  30922949.1075863688243.JavaMail.evans@thyme   

                                    Date                     From  \
0  Mon, 14 May 2001 16:39:00 -0700 (PDT)  phillip.allen@enron.com   
1   Fri, 4 May 2001 13:51:00 -0700 (PDT)  phillip.allen@enron.com   
2  Wed, 18 Oct 2000 03:00:00 -0700 (PDT)  phillip.allen@enron.com   
3  Mon, 23 Oct 2000 06:13:00 -0700 (PDT)  phillip.allen@enron.com   
4  Thu, 31 Aug 2000 05:07:00 -0700 (PDT)  phillip.allen@enron.com   

                        To    Subject   Cc  Mime-Version  \
0     tim.belden@enron.com        NaN  NaN           1.0   
1  john.lavorato@enron.com        Re:  NaN           1.0   
2   leah.arsdall@enron.com   Re: test  NaN           1.0   
3    randall.gay@enron.com        NaN  NaN  

In [374]:
emails_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517401 entries, 0 to 517400
Data columns (total 18 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Message-ID                 495554 non-null  object 
 1   Date                       495554 non-null  object 
 2   From                       495554 non-null  object 
 3   To                         495554 non-null  object 
 4   Subject                    478886 non-null  object 
 5   Cc                         124262 non-null  object 
 6   Mime-Version               495554 non-null  float64
 7   Content-Type               495554 non-null  object 
 8   Content-Transfer-Encoding  495554 non-null  object 
 9   Bcc                        126416 non-null  object 
 10  X-From                     495554 non-null  object 
 11  X-To                       495554 non-null  object 
 12  X-cc                       127172 non-null  object 
 13  X-bcc                      16

In [375]:
emails_df.isnull().sum()

Message-ID                    21847
Date                          21847
From                          21847
To                            21847
Subject                       38515
Cc                           393139
Mime-Version                  21847
Content-Type                  21847
Content-Transfer-Encoding     21847
Bcc                          390985
X-From                        21847
X-To                          21847
X-cc                         390229
X-bcc                        517233
X-Folder                      21847
X-Origin                      21847
X-FileName                    22394
Message-Body                  21848
dtype: int64

In [376]:
emails_df.dropna(how='all', inplace=True)
emails_df.dropna(subset=['Message-Body'], inplace=True)

In [377]:
# drop columns with lots of missing values
print(emails_df['Bcc'].isnull().mean() * 100, '% of X-bcc is empty')
print(emails_df['Cc'].isnull().mean() * 100, '% of X-cc is empty')
print(emails_df['X-bcc'].isnull().mean() * 100, '% of X-bcc is empty')
print(emails_df['X-cc'].isnull().mean() * 100, '% of X-cc is empty')

emails_df.drop(columns=['Bcc', 'Cc', 'X-bcc', 'X-cc'], inplace=True)
print('remaining cols: ', emails_df.columns)

74.4899132887905 % of X-bcc is empty
74.92457920747125 % of X-cc is empty
99.96609847988005 % of X-bcc is empty
74.33735644825074 % of X-cc is empty
remaining cols:  Index(['Message-ID', 'Date', 'From', 'To', 'Subject', 'Mime-Version',
       'Content-Type', 'Content-Transfer-Encoding', 'X-From', 'X-To',
       'X-Folder', 'X-Origin', 'X-FileName', 'Message-Body'],
      dtype='object')


In [378]:
# Impute with No Subject
emails_df['Subject'].fillna('No Subject', inplace=True)

In [379]:
# Drop columns that are not useful for prediction
emails_df = emails_df.drop(columns=['Message-ID', 'Mime-Version', 'Content-Type', 'Content-Transfer-Encoding', 'X-Folder', 'X-Origin', 'X-FileName'])

In [380]:
emails_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 495553 entries, 0 to 517400
Data columns (total 7 columns):
 #   Column        Non-Null Count   Dtype 
---  ------        --------------   ----- 
 0   Date          495553 non-null  object
 1   From          495553 non-null  object
 2   To            495553 non-null  object
 3   Subject       495553 non-null  object
 4   X-From        495553 non-null  object
 5   X-To          495553 non-null  object
 6   Message-Body  495553 non-null  object
dtypes: object(7)
memory usage: 30.2+ MB


In [381]:
print(emails_df['Message-Body'][::16000])

0                                 Here is our forecast\n\n 
16514     in\n\n\n   \n\n\nFrom:  Bryan Hull            ...
33080     The Commission issued an order on May 8, 2001 ...
51134     Hey Dirk,\n\n Could you tell me where I would ...
67703     713 646-8525\n\n -----Original Message-----\nF...
84182      The Ron Brown Scholarships are available for ...
101003    The volume for the 24th is in error.  It is re...
117344    ---------------------- Forwarded by Drew Fossu...
133683    Hey, I'm too tired to think so I started going...
150256    Deal 457758.1 for 11/14/00, HE 7 shows 25mw @ ...
167207    Myself,  being of Mexican descent there is no ...
183877    Susan/Tana,\n\nWe have recently traded a deal ...
200040    ---------------------- Forwarded by Vince J Ka...
216488    Our proposal was accepted. Dust off your San F...
233049    I'll send it back on Tuesday.\n\n\n\n\nJose Be...
250105    Attached please find two documents for Friday'...
266886    Wait until Louise gets back.  

In [382]:
print(emails_df['Subject'][9], emails_df['Message-Body'][9])

FW: fixed forward or other Collar floor gas price terms ---------------------- Forwarded by Phillip K Allen/HOU/ECT on 10/16/2000 
01:42 PM ---------------------------


"Buckner, Buck" <buck.buckner@honeywell.com> on 10/12/2000 01:12:21 PM
To: "'Pallen@Enron.com'" <Pallen@Enron.com>
cc:  
Subject: FW: fixed forward or other Collar floor gas price terms


Phillip,

> As discussed  during our phone conversation, In a Parallon 75 microturbine
> power generation deal for a national accounts customer, I am developing a
> proposal to sell power to customer at fixed or collar/floor price. To do
> so I need a corresponding term gas price for same. Microturbine is an
> onsite generation product developed by Honeywell to generate electricity
> on customer site (degen). using natural gas. In doing so,  I need your
> best fixed price forward gas price deal for 1, 3, 5, 7 and 10 years for
> annual/seasonal supply to microturbines to generate fixed kWh for
> customer. We have the opportunity to sel

In [383]:
emails_df['Message-Body']

0                                 Here is our forecast\n\n 
1         Traveling to have a business meeting takes the...
2                            test successful.  way to go!!!
3         Randy,\n\n Can you send me a schedule of the s...
4                       Let's shoot for Tuesday at 11:45.  
                                ...                        
517396    This is a trade with OIL-SPEC-HEDGE-NG (John L...
517397    Some of my position is with the Alberta Term b...
517398    2\n\n -----Original Message-----\nFrom: \tDouc...
517399    Analyst\t\t\t\t\tRank\n\nStephane Brodeur\t\t\...
517400    i think the YMCA has a class that is for peopl...
Name: Message-Body, Length: 495553, dtype: object

In [384]:
# Find the emails that have common reply or forward patterns
# -+\s*(Original|Forwarded)
def find_reply_forward(msg_body):

    if not isinstance(msg_body, str):
        return False

    pattern = re.compile(
        r"""
        (?:-+\s*(?:Original|Forwarded)) # forwarded
        |On\s+[^\n]+wrote: # replies
        |>\s*(?:From|Sent|To): # Quoted replies
        |Note:\s*forwarded\s*message\s*attached  # Forwarded note
        |Begin\s+forwarded\s+message
        |Message\s+forwarded
        """,
        re.IGNORECASE | re.VERBOSE | re.DOTALL)
    return bool(pattern.search(msg_body))

emails_df['has_reply_forward_in_msg'] = emails_df['Message-Body'].apply(find_reply_forward)


In [385]:
emails_df['has_reply_forward_in_msg'].value_counts()

False    320509
True     175044
Name: has_reply_forward_in_msg, dtype: int64

In [386]:
# Remove the emails that have reply or forward
emails_df = emails_df[emails_df['has_reply_forward_in_msg'] == False]
emails_df['has_reply_forward_in_msg'].value_counts()
emails_df.drop(columns=['has_reply_forward_in_msg'], inplace=True)


In [387]:
# trim the spaces
def trim_space_remove_image(msg_body):
    if not isinstance(msg_body, str):
        return ''
    else:
        msg_body = quopri.decodestring(msg_body).decode('utf-8', errors='ignore')
        msg_body = re.sub(r'\[IMAGE\]', '', msg_body)
        msg_body = msg_body.strip()
        msg_body = re.sub(r'\n+', '\n', msg_body)
        return msg_body

emails_df['Message-Body'] = emails_df['Message-Body'].apply(trim_space_remove_image)

In [388]:
emails_df.describe().T

,count,unique,top,freq
Date,320509,138485,"Mon, 31 Dec 1979 16:00:00 -0800 (PST)",281
From,320509,17615,pete.davis@enron.com,9147
To,320509,39913,pete.davis@enron.com,9146
Subject,320509,103132,No Subject,14810
X-From,320509,23588,Enron Announcements,8525
X-To,320509,48608,pete.davis@enron.com,5334
Message-Body,320509,148000,We've updated the Merger Q&A document on our E...,110


In [389]:
# drop duplicates
print(emails_df['Message-Body'].value_counts())
emails_df.drop_duplicates(subset=['Message-Body'], inplace=True)
emails_df.reset_index(drop=True, inplace=True)

We've updated the Merger Q&A document on our Enron Updates site ( <http://home.enron.com/updates/mergerQA.html>), as a result of the many questions you've had concerning the merger between Enron and Dynegy. Questions addressed include those about Enron stock options, benefits and immigration status. Please stay tuned for additional updates.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [390]:
emails_df.describe().T

,count,unique,top,freq
Date,148000,134204,"Mon, 31 Dec 1979 16:00:00 -0800 (PST)",175
From,148000,17392,pete.davis@enron.com,4369
To,148000,39093,pete.davis@enron.com,4369
Subject,148000,101674,No Subject,6807
X-From,148000,23087,"Davis, Pete </O=ENRON/OU=NA/CN=RECIPIENTS/CN=P...",2813
X-To,148000,47141,"Davis, Pete </O=ENRON/OU=NA/CN=RECIPIENTS/CN=P...",3381
Message-Body,148000,148000,Here is our forecast,1


In [391]:
emails_df["Message-Body"][::4000]

0                                      Here is our forecast
4000      I guess that feeling is similar to the one you...
8000      A new secure Web-Based Rate and Currency Repor...
12000     To Market Participants,\nThe Premier of the ne...
16000     The following expense report is ready for appr...
20000     <<Re remove cat urine from an area rug.htm>>\n...
24000     I assume this will be discussed at tomorrow's ...
28000     Start Date: 10/17/01; HourAhead hour: 9;  Hour...
32000     Please advise should you have any questions co...
36000     Once Susan gets the last few pieces of info, e...
40000     Learn Technical Analysis\nTwo Full Days, Decem...
44000     I need everyone to close out of Lotus Notes an...
48000     In Between Holidays, Post Budget, Post Electio...
52000     Platts Energy Bulletin\nThe daily Energy Bulle...
56000     Mary:\nEnclosed is the Counterparty form that ...
60000     Stinson,\nI think the gamma will flow into V@R...
64000     Now available at the New York 

In [392]:
print(emails_df["Message-Body"][80000])

Upgrades   DownGrades   Coverage Initiated   Coverage Reiterated   Stock Splits   Buybacks   Dividends   Pos Pre-Announce   Neg Pre-Announce   Pos Surprises   Neg Surprises   Earnings Revisions   IPO - Lockup Periods   IPO - Latest Pricing   IPO - Quite Periods   IPO - Postponements   IPO - Withdrawals   IPO - Latest Filings                Unsubscribe   Update  my Membership / Profile   Forgot  Username / Password   Add  / Edit Alerts   View  My Alerts                  	  As requested, your News Alert for QCOM  follows from EquityAlert.com.     Multimedia Available: QUALCOMM Introduces Technology to Enhance Aviation Safety Services   Oct 29, 2001 (BUSINESS WIRE) -- QUALCOMM Incorporated (Nasdaq:QCOM), pioneer and world leader of Code Division Multiple Access (CDMA) digital wireless technology, successfully demonstrated a set of aviation safety solutions using the Globalstar(tm) Satellite Communications System and onboard hardware incorporating CDMA technology. The MDSS Globalstar Commu

In [393]:
def is_low_value(msg_body, min_word_count=3):
    if not isinstance(msg_body, str):
        return False
    if len(msg_body.split()) < min_word_count:
        return True
    return False

emails_df = emails_df[~emails_df['Message-Body'].apply(is_low_value)]

In [394]:
emails_df['Message-Body'][::6000]

0                                      Here is our forecast
6133      Don,\nPlease advise of your interest in Alex's...
12203     On April 24, 2001, the NYISO deployed code cha...
18369     April 16, 2001\nPR UPDATE\nTo: ???IEP PR Commi...
24459     Dear Former TheStreet.com Reader:\nNot too lon...
30556     Margaret,\nPlease send them to the Calgary off...
36604     System Outage Notification\nOutage Description...
42730     Ina,\nI need to add my son, Clark Watson, to m...
48776     Per your request, here are the GTC's for EGLI ...
54829     We have received an executed Assignment and As...
60915     Slava,\nCan we invite Maureen Raymond ( a memb...
66994     This is the final version that was taken to th...
73077     John,\nIn reviewing the remaining schedule C, ...
79178     WELCOME - Vol. 6 No. 44\nTIMELY INVESTMENT INF...
85285     Hi Suzanne,\nI haven't seen a revised flight s...
91386     Attached is an Excel file containing the Nymex...
97436     The meeting will be in Jeff's 

In [395]:
emails_df.to_csv('../data/02_emails_handled_missing_reply_forward.csv', index=False)